In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os

In [2]:
df = pd.read_excel('/workspace/results/Thyrocyte AI Clustering Evaluation Form (Responses).xlsx', sheet_name='Form Responses 1')

In [3]:
data = []
for root, dirs, files in os.walk('/workspace/results/clustering_steps'):
    for file in files:
        level = root.split('/')[-1]
        image_a = file.split('_')[-1].split('.')[0] + "-A"
        image_b = file.split('_')[-1].split('.')[0] + "-B"
        
        print(f"Level: {level}, Image A: {image_a}")
        for row in df[image_a]:
            print(f"\tPathologist Rating: {row}")
            data.append({'Level': level, 'Image': image_a, 'Rating': row, "Step": "2-Step Clustering"})
        print(f"Level: {level}, Image B: {image_b}")
        for row in df[image_b]:
            print(f"\tPathologist Rating: {row}")
            data.append({'Level': level, 'Image': image_b, 'Rating': row, "Step": "3-Step Clustering"})
df_ratings = pd.DataFrame(data)

Level: LEVEL_IV, Image A: LS-282-A
	Pathologist Rating: 3
	Pathologist Rating: 2
	Pathologist Rating: 1
	Pathologist Rating: 3
Level: LEVEL_IV, Image B: LS-282-B
	Pathologist Rating: 3
	Pathologist Rating: 4
	Pathologist Rating: 1
	Pathologist Rating: 3
Level: LEVEL_IV, Image A: LS-494-A
	Pathologist Rating: 4
	Pathologist Rating: 3
	Pathologist Rating: 4
	Pathologist Rating: 3
Level: LEVEL_IV, Image B: LS-494-B
	Pathologist Rating: 4
	Pathologist Rating: 4
	Pathologist Rating: 4
	Pathologist Rating: 4
Level: LEVEL_IV, Image A: LS-503-A
	Pathologist Rating: 4
	Pathologist Rating: 3
	Pathologist Rating: 4
	Pathologist Rating: 3
Level: LEVEL_IV, Image B: LS-503-B
	Pathologist Rating: 4
	Pathologist Rating: 4
	Pathologist Rating: 4
	Pathologist Rating: 4
Level: LEVEL_IV, Image A: LS-395-A
	Pathologist Rating: 4
	Pathologist Rating: 1
	Pathologist Rating: 3
	Pathologist Rating: 3
Level: LEVEL_IV, Image B: LS-395-B
	Pathologist Rating: 4
	Pathologist Rating: 1
	Pathologist Rating: 3
	Pathol

In [4]:
df_ratings["Smear"] = df_ratings["Image"].str.replace("-A", "").str.replace("-B", "")

In [5]:
df_ratings

,Level,Image,Rating,Step,Smear
0,LEVEL_IV,LS-282-A,3,2-Step Clustering,LS-282
1,LEVEL_IV,LS-282-A,2,2-Step Clustering,LS-282
2,LEVEL_IV,LS-282-A,1,2-Step Clustering,LS-282
3,LEVEL_IV,LS-282-A,3,2-Step Clustering,LS-282
4,LEVEL_IV,LS-282-B,3,3-Step Clustering,LS-282
...,...,...,...,...,...
747,LEVEL_II,LS-541-A,4,2-Step Clustering,LS-541
748,LEVEL_II,LS-541-B,4,3-Step Clustering,LS-541
749,LEVEL_II,LS-541-B,4,3-Step Clustering,LS-541
750,LEVEL_II,LS-541-B,4,3-Step Clustering,LS-541


In [6]:
paired = df_ratings.pivot_table(
    index=["Smear"],
    columns="Step",
    values="Rating",
    aggfunc="mean"
).dropna()

In [7]:
paired

Step,2-Step Clustering,3-Step Clustering
Smear,,
LS-021,3.25,3.50
LS-068,2.50,3.00
LS-075,3.00,3.00
LS-099,4.00,4.00
LS-113,3.00,2.75
LS-168,3.50,3.75
LS-172,3.50,4.00
LS-188,3.50,3.50
LS-218,3.50,3.75


In [8]:
from scipy.stats import wilcoxon

stat, p = wilcoxon(
    paired["2-Step Clustering"],
    paired["3-Step Clustering"]
)

print("Wilcoxon statistic:", stat)
print("p-value:", p)

Wilcoxon statistic: 15.0
p-value: 0.00014223444092052528


The Wilcoxon signed-rank test indicates that the ratings assigned to clusters generated using the 3-step aggregation method were significantly higher than those generated using the 2-step aggregation method.

In [9]:
paired["diff"] = paired["2-Step Clustering"] - paired["3-Step Clustering"]

In [10]:
paired

Step,2-Step Clustering,3-Step Clustering,diff
Smear,,,
LS-021,3.25,3.50,-0.25
LS-068,2.50,3.00,-0.50
LS-075,3.00,3.00,0.00
LS-099,4.00,4.00,0.00
LS-113,3.00,2.75,0.25
LS-168,3.50,3.75,-0.25
LS-172,3.50,4.00,-0.50
LS-188,3.50,3.50,0.00
LS-218,3.50,3.75,-0.25


In [11]:
paired["Preference"] = 0
paired.loc[paired["diff"] > 0, "Preference"] = 1
paired.loc[paired["diff"] < 0, "Preference"] = -1

In [12]:
wins_2 = sum(paired["Preference"] == 1)
wins_3 = sum(paired["Preference"] == -1)
ties = sum(paired["Preference"] == 0)

print("2-Step preferred:", wins_2)
print("3-Step preferred:", wins_3)
print("Ties:", ties)

2-Step preferred: 3
3-Step preferred: 20
Ties: 24


Interpretation:
 - In 20 smears, experts preferred 3-step
 - In 3 smears, experts preferred 2-step
 - In 24 smears, both methods were equal

In simple smears, both methods produce the same clusters → ties
In complex smears, methods differ → 3-step usually preferred

Based on the data,
 - Total smears = 47
     - ~51% → same clusters
     - ~43% → 3-step better
     - ~6%  → 2-step better

In about half of the smears, additional aggregation does not change cluster structure. However, when differences occur, the 3-step aggregation method is more frequently preferred, suggesting that deeper spatial aggregation is beneficial primarily in more complex smear structures.

H0: Both methods are equally likely to be preferred
    P(3-step preferred) = 0.5

H1: The methods are not equally likely to be preferred

But based on the result:
 - 3-step clustering is preferred over 2-step clustering.
 - Hence, x = 3-step clustering

and answering the test:
 - If both methods are equally good (50/50), what is the probability of seeing 20 or more preferences for 3-step out of 23 comparisons?

In [16]:
from scipy.stats import binomtest

n = wins_2 + wins_3
x = wins_3
result = binomtest(x, n, p=0.5)

print(result)

BinomTestResult(k=20, n=23, alternative='two-sided', statistic=0.8695652173913043, pvalue=0.00048828125)


Assuming both clustering methods are equally likely to be preferred (50% each), the probability of observing 20 or more preferences for the 3-step method out of 23 comparisons is 0.0488%. Since this probability is very small (p < 0.001), we reject the null hypothesis and conclude that experts significantly preferred the 3-step clustering method over the 2-step clustering method.


The binomial test indicates that experts preferred the 3-step clustering method significantly more often than the 2-step clustering method when differences between methods were observed.

In [14]:
df_ratings.to_csv('/workspace/results/Clustering Evaluation.csv', index=False)

A Wilcoxon signed-rank test showed that the 3-step clustering method received significantly higher ratings than the 2-step clustering method (p < 0.001). Preference analysis showed that experts preferred the 3-step clustering results in 20 smears, compared to only 3 smears where the 2-step clustering was preferred, while 24 smears showed no difference between methods. A binomial test confirmed that the preference for the 3-step method was statistically significant (p < 0.001). These findings suggest that while both methods produce similar cluster structures in simpler smears, additional aggregation in the 3-step method improves cluster meaningfulness in more complex smear structures.

The high number of ties indicates that in many smears, particularly those with simple spatial distributions, both aggregation methods produce similar cluster structures. However, when differences between methods occur, the 3-step aggregation method is more frequently preferred and receives higher ratings. This suggests that additional hierarchical aggregation is beneficial primarily in smears with more complex spatial distributions, where further spatial consolidation improves the representation of meaningful cell clusters.

| Analysis          | Purpose                  |
| ----------------- | ------------------------ |
| Mean ratings      | Overall comparison       |
| Wilcoxon          | Paired rating difference |
| Preference counts | Practical comparison     |
| Binomial          | Preference significance  |


Expert Review of Spatial Aggregation Results
Following expert review using the test data, spatial cluster visualizations generated by the two aggregation approaches were evaluated based on perceived clinical relevance, spatial coherence, and representation of meaningful thyrocyte groupings. Expert ratings were analyzed using paired statistical testing and preference analysis to determine whether one aggregation strategy produced more clinically meaningful cluster representations. 

### Table X. Summary of Expert Evaluation Results

| Analysis                              | Result    |
|--------------------------------------|----------:|
| Wilcoxon Signed-Rank Statistic       | 15.0      |
| Wilcoxon p-value                     | 0.000142  |
| Smears where 3-Step preferred        | 20        |
| Smears where 2-Step preferred        | 3         |
| Smears with no difference (ties)     | 24        |
| Total smears evaluated               | 47        |
| Binomial test p-value                | 0.000488  |

A Wilcoxon signed-rank test was conducted to compare expert ratings between the 2-step and 3-step aggregation methods. The test showed a statistically significant difference between the ratings assigned to the two methods (p < 0.001), indicating that clusters generated using the 3-step aggregation method were rated significantly higher than those generated using the 2-step aggregation method.
Preference analysis was performed by comparing the mean expert rating for each smear between the two aggregation methods. The method with the higher rating for each smear was considered the preferred method, while equal ratings were recorded as ties.

Results showed that:

 - Experts preferred the 3-step aggregation method in 20 smears 
 - Experts preferred the 2-step aggregation method in 3 smears 
 - 24 smears showed no difference between methods 
 
Out of the 47 smears evaluated:
 - ~51% showed no difference between methods 
 - ~43% favored the 3-step aggregation 
 - ~6% favored the 2-step aggregation 
 
A binomial test conducted on the non-tie comparisons (n = 23) showed that the preference for the 3-step method was statistically significant (p < 0.001), indicating that experts preferred the 3-step aggregation method significantly more often than the 2-step aggregation method when differences between methods were observed.

The results support H3 (Clinical Interpretability), which hypothesized that expert pathologists would judge the multi-stage (3-step) cluster representations as more clinically plausible and closer to their annotated groupings than the direct (2-step) aggregation strategy.
The Wilcoxon signed-rank test demonstrated that the 3-step aggregation method received significantly higher ratings overall, indicating that experts generally perceived the multi-stage aggregation clusters as more clinically meaningful. Preference analysis further showed that when differences between methods occurred, experts overwhelmingly preferred the 3-step aggregation method.
An important observation from the results is the high number of ties (24 out of 47 smears). This indicates that in many smears—particularly those with simpler spatial distributions—both aggregation methods produced similar cluster structures, resulting in no perceived difference in clinical meaningfulness. This suggests that additional aggregation stages may not significantly alter cluster structure in simpler smear configurations.

However, when differences between methods were observed, the 3-step aggregation method was more frequently preferred and received higher ratings. This suggests that additional hierarchical aggregation is beneficial primarily in smears with more complex spatial distributions, where further spatial consolidation improves the representation of meaningful thyrocyte clusters.
Overall, these findings suggest that while both aggregation methods can produce similar cluster structures in simpler smear cases, the multi-stage (3-step) aggregation approach provides improved spatial cluster representation in more complex smear structures. The results therefore support the hypothesis that multi-stage spatial aggregation produces cluster structures that more closely resemble clinically meaningful thyrocyte groupings compared to direct two-step aggregation.